# Real-photo frozen pipeline: part B

Private RGB-only inference using unchanged source from commit 5b78942. All 29 scenes are covered across parts A and B. Each scene uses its first five views and its remaining two or three views as separate app-compatible inspections. Failures and review-required decisions are retained. No anomaly labels are mounted; no thresholds, model weights or decision rules are tuned.


In [1]:
from pathlib import Path
import hashlib, json, subprocess, sys, zipfile
def sha(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()
expected_manifest = 'c9e3b280fbccf6dc0fbe0b0972b52a936efb038733b999a35951fe72eaaf9d7a'
candidates = [p for p in Path('/kaggle/input').rglob('batch-manifest.json') if sha(p) == expected_manifest]
if not candidates:
    archives = list(Path('/kaggle/input').rglob('batch.zip'))
    assert len(archives) == 1, 'Cannot resolve the unique private RGB package'
    target = Path('/tmp/vpi-runtime-realphoto-input')
    target.mkdir(exist_ok=False)
    with zipfile.ZipFile(archives[0]) as archive:
        names = archive.namelist()
        assert len(names) == len(set(names)), 'Duplicate archive members'
        assert len(names) < 10000 and sum(i.file_size for i in archive.infolist()) < 3 * 1024**3
        for info in archive.infolist():
            name = info.filename
            assert not Path(name).is_absolute() and '..' not in Path(name).parts and '\\' not in name
            assert (info.external_attr >> 16) & 0o170000 != 0o120000, 'Archive symlink refused'
        archive.extractall(target)
    candidates = [p for p in target.rglob('batch-manifest.json') if sha(p) == expected_manifest]
assert len(candidates) == 1, 'Ambiguous or changed batch manifest'
root = candidates[0].parent
manifest = json.loads(candidates[0].read_text())
assert sha(root / 'run_batch.py') == manifest['runner_sha256']
print(json.dumps({k: manifest[k] for k in ['source_commit', 'scene_count', 'image_count', 'case_count', 'view_rule']}, indent=2))


{
  "source_commit": "5b78942e29fe7186458c6a13306eeba2657fc3e6",
  "scene_count": 29,
  "image_count": 231,
  "case_count": 58,
  "view_rule": "capture-order first five, then remaining two or three"
}


In [2]:
output = Path('/kaggle/working/real-photo-part-b')
command = [sys.executable, '-S', '-B', str(root / 'run_batch.py'), '--root', str(root),
           '--output', str(output), '--shard', '1']
child = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in child.stdout:
    print(line.rstrip(), flush=True)
status = child.wait()
if status:
    raise RuntimeError('Batch failed; retain output receipts and logs for recovery')


setup: 30 seconds
setup: 60 seconds
setup: 90 seconds
setup: 120 seconds
setup: 150 seconds
setup: 180 seconds
setup: 210 seconds
setup: 240 seconds
setup: 270 seconds
setup: 300 seconds
preflight: 30 seconds
preflight: 60 seconds
preflight: 90 seconds
Starting S016-A, 5 views
S016-A: 30 seconds
S016-A: 60 seconds
{"case_id": "S016-A", "elapsed_seconds": 78.026, "observations": 20, "review_state": "review_required", "selected_identity_id": "winner-consensus-002", "status": "complete", "view_count": 5}
Starting S016-B, 3 views
S016-B: 30 seconds
S016-B: 60 seconds
{"case_id": "S016-B", "elapsed_seconds": 58.936, "observations": 12, "review_state": "review_required", "selected_identity_id": "winner-consensus-001", "status": "complete", "view_count": 3}
Starting S017-A, 5 views
S017-A: 30 seconds
S017-A: 60 seconds
{"case_id": "S017-A", "elapsed_seconds": 76.981, "observations": 15, "review_state": "review_required", "selected_identity_id": "winner-consensus-001", "status": "complete", "v

In [3]:
summary = json.loads((output / 'run-summary.json').read_text())
assert summary['cases_attempted'] == summary['cases_expected']
assert summary['manifest_sha256'] == expected_manifest
assert (output / 'run-lock.json').is_file()
print(json.dumps(summary, indent=2))
print('All scheduled inspections attempted. Accuracy and Assisted evaluation remain separate review tasks.')


{
  "accuracy_evaluated": false,
  "assisted_executed": false,
  "cases_attempted": 28,
  "cases_complete": 28,
  "cases_expected": 28,
  "cases_failed": 0,
  "manifest_sha256": "c9e3b280fbccf6dc0fbe0b0972b52a936efb038733b999a35951fe72eaaf9d7a",
  "receipts": [
    {
      "case_id": "S016-A",
      "elapsed_seconds": 78.026,
      "observations": 20,
      "review_state": "review_required",
      "selected_identity_id": "winner-consensus-002",
      "status": "complete",
      "view_count": 5
    },
    {
      "case_id": "S016-B",
      "elapsed_seconds": 58.936,
      "observations": 12,
      "review_state": "review_required",
      "selected_identity_id": "winner-consensus-001",
      "status": "complete",
      "view_count": 3
    },
    {
      "case_id": "S017-A",
      "elapsed_seconds": 76.981,
      "observations": 15,
      "review_state": "review_required",
      "selected_identity_id": "winner-consensus-001",
      "status": "complete",
      "view_count": 5
    },
    {


In [4]:
archive_path = output.parent / (output.name + '-evidence.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
    for path in sorted(output.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(output).as_posix())
receipt = {'schema': 'vpi-real-photo-evidence-export/v1',
           'manifest_sha256': expected_manifest, 'zip_sha256': sha(archive_path),
           'zip_bytes': archive_path.stat().st_size, 'run_lock_sha256': sha(output / 'run-lock.json'),
           'cases_attempted': summary['cases_attempted'], 'cases_complete': summary['cases_complete'],
           'cases_failed': summary['cases_failed'], 'views_attempted': summary['views_attempted']}
(output.parent / (output.name + '-evidence-receipt.json')).write_text(json.dumps(receipt, indent=2))
print(json.dumps(receipt, indent=2))


{
  "schema": "vpi-real-photo-evidence-export/v1",
  "manifest_sha256": "c9e3b280fbccf6dc0fbe0b0972b52a936efb038733b999a35951fe72eaaf9d7a",
  "zip_sha256": "d973d8c42e5b063371c9d2400fe96654904f6cb6b6ec7ae2e3fd3301e991c8fb",
  "zip_bytes": 3279788631,
  "run_lock_sha256": "a20b283774e5580ef0a334b65c5f103cedfee784ae3fb1ef6c6c976ae43bf666",
  "cases_attempted": 28,
  "cases_complete": 28,
  "cases_failed": 0,
  "views_attempted": 112
}
